# A3a — one network for legs A, B and C

This notebook only **loads** what the scripts wrote. Nothing here recomputes a
network, a reference or a score; the numbers come from `results/` and the
figures from `figures/`.

| script | output |
|---|---|
| `build_dataset.py` | `results/general_legs.npz`, `results/dataset_meta.json` |
| `_shared/train.py` (26 farm jobs) | `results/<tag>.json`, `<tag>.pt`, `<tag>_history.csv` |
| `aggregate.py` | `results/summary.csv`, `results/by_leg.csv`, `results/stage_errors.csv` |
| `plot.py` | the three figures |


In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

meta = json.load(open("results/dataset_meta.json"))
print("training states:", meta["n_train_chosen"], "->", meta["counts"])
print("per leg:", meta["per_leg"]["train"])
pd.DataFrame(meta["timing_table"])

## How long one L-BFGS restart takes, and therefore how big the training set is

In [ ]:
pd.DataFrame(meta["momentum_mix"]["train"]).T

The builder caps the pool with a plain random permutation, so the momentum
mix above is the training set's own mix, recorded rather than imposed.

## The runs

In [ ]:
summary = pd.read_csv("results/summary.csv")
summary[["tag", "mode", "seed", "width", "restarts", "converged", "final_loss",
         "test_endpoint_med_um", "test_endpoint_p95_um", "wall_s"]].sort_values(
    ["width", "mode", "seed"])

## Leg by leg, against the exact-scheme ceiling

In [ ]:
by_leg = pd.read_csv("results/by_leg.csv")
t = by_leg[(by_leg.split == "test") & (by_leg.band == "all") & by_leg.converged]
tab = t.pivot_table(index=["leg"], columns=["width", "mode"],
                    values="endpoint_med_um", aggfunc="median")
tab["straight_um"] = t.groupby("leg").straight_med_um.median()
tab["ceiling_q8_um"] = t.groupby("leg").ceiling_leg_um.first()
tab

In [ ]:
display(Image("figures/error_by_leg_and_momentum.png"))

## The error along the step, not only at its end

In [ ]:
display(Image("figures/stage_errors.png"))

## The cross-magnet leg: frozen leg vs one network for everything

In [ ]:
display(Image("figures/frozen_vs_general.png"))